## Coding Attention mechanisms


### interesting topic :   Bahdanau attention
- self-attention mechanism inspired by the Bahdanau attention mechanism.
- Self-attention is a mechanism in transformers used to compute
more efficient input representations
- The “self” in self-attention
In self-attention, the “self” refers to the mechanism’s ability to compute attention
weights by relating different positions within a single input sequence. It assesses and
learns the relationships and dependencies between various parts of the input itself,
such as words in a sentence or pixels in an image.

### A simple self-attention mechanism without trainable weights

- ω,attention scores
- z context vector 
- alpha attention weight after normalization of attention score

In [1]:
import  torch
inputs =torch.tensor(
[[0.43, 0.15, 0.89], # Your (x^1)
[0.55, 0.87, 0.66], # journey (x^2)
[0.57, 0.85, 0.64], # starts (x^3)
[0.22, 0.58, 0.33], # with (x^4)
[0.77, 0.25, 0.10], # one (x^5)
[0.05, 0.80, 0.55]] # step (x^6)
)

In [2]:
inputs.shape

torch.Size([6, 3])

In [4]:
query = inputs[1] # Your (x^1)
attn_scores_2 = torch.empty(inputs.shape[0])

for i , x_i in enumerate(inputs):
    print(f"Calculating attention score for x^{i+1} with query x^2")
    print(f"Query: {query}")
    print(f"Key: {x_i}")
    attn_scores_2[i] = torch.dot(query,x_i)
    print(f"Attention score: {attn_scores_2[i]}")
    print("-"*50)

Calculating attention score for x^1 with query x^2
Query: tensor([0.5500, 0.8700, 0.6600])
Key: tensor([0.4300, 0.1500, 0.8900])
Attention score: 0.9544000625610352
--------------------------------------------------
Calculating attention score for x^2 with query x^2
Query: tensor([0.5500, 0.8700, 0.6600])
Key: tensor([0.5500, 0.8700, 0.6600])
Attention score: 1.4950001239776611
--------------------------------------------------
Calculating attention score for x^3 with query x^2
Query: tensor([0.5500, 0.8700, 0.6600])
Key: tensor([0.5700, 0.8500, 0.6400])
Attention score: 1.4754000902175903
--------------------------------------------------
Calculating attention score for x^4 with query x^2
Query: tensor([0.5500, 0.8700, 0.6600])
Key: tensor([0.2200, 0.5800, 0.3300])
Attention score: 0.8434000015258789
--------------------------------------------------
Calculating attention score for x^5 with query x^2
Query: tensor([0.5500, 0.8700, 0.6600])
Key: tensor([0.7700, 0.2500, 0.1000])
Attenti

- the dot product is a measure of similarity
because it quantifies how closely two vectors are aligned: a higher dot product indi-
cates a greater degree of alignment or similarity between the vectors.
 - This normalization is a convention that is useful for interpre-
tation and maintaining training stability in an LLM.

In [ ]:
attn_weights_2_temp = attn_scores_2 / attn_scores_2.sum() # better to use softmax function for normalization but here we are using simple sum for normalization
print(f"Attention weights (unnormalized): {attn_scores_2}")
print(f"Attention weights (normalized): {attn_weights_2_temp}")

Attention weights (unnormalized): tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])
Attention weights (normalized): tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])


In [6]:
def softmax_naive(x):
    exp_x = torch.exp(x)
    return exp_x / exp_x.sum(dim=0)


attn_weights_2_naive = softmax_naive(attn_scores_2)
print(f"Attention weights (softmax normalized): {attn_weights_2_naive}")
print(f"Sum of unnormalized attention weights: {attn_scores_2.sum()}")
print(f"Sum of normalized attention weights (simple sum): {attn_weights_2_temp.sum()}")
print(f"Sum of normalized attention weights (softmax): {attn_weights_2_naive.sum()}")

Attention weights (softmax normalized): tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum of unnormalized attention weights: 6.561699867248535
Sum of normalized attention weights (simple sum): 1.0000001192092896
Sum of normalized attention weights (softmax): 1.0


- softmax function ensures that the attention weights are always positive. 

- makes the output interpretable as probabilities or relative importance, where higher weights indicate greater importance

In [7]:
attn_weights_2 =torch.softmax(attn_scores_2, dim=0)
print(f"Attention weights (softmax normalized using PyTorch): {attn_weights_2}")
print(f"Sum of normalized attention weights (softmax using PyTorch): {attn_weights_2.sum()}")

Attention weights (softmax normalized using PyTorch): tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum of normalized attention weights (softmax using PyTorch): 1.0


In [ ]:
query = inputs[1]# Your (x^1)
context_vector_2 = torch.zeros(query.shape) # initialize context vector with zeros
for i , x_i in enumerate(inputs):
    context_vector_2 += attn_weights_2[i]*x_i
    print(context_vector_2)
    
print("-"*50)
print(f"final context_vector: { context_vector_2}" )

tensor([0.0596, 0.0208, 0.1233])
tensor([0.1904, 0.2277, 0.2803])
tensor([0.3234, 0.4260, 0.4296])
tensor([0.3507, 0.4979, 0.4705])
tensor([0.4340, 0.5250, 0.4813])
tensor([0.4419, 0.6515, 0.5683])
--------------------------------------------------
final context_vector: tensor([0.4419, 0.6515, 0.5683])


### Computing attention weights for all input tokens

1. Compute the attention scores as
dot products between the inputs.

2. The attention weights are a normalized
version of the attention scores.

3. The context vectors are computed as
a weighted sum over the inputs.

In [10]:
attn_scores = torch.empty(6 ,6)
for i , x_i in enumerate(inputs):
    for j , x_j in enumerate(inputs):
        attn_scores[ i , j] = torch.dot(x_i, x_j)
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [11]:
## for loops are generally slow, and we can achieve the same results using matrix multiplication

attn_scores = inputs @ inputs.T
print(attn_scores)

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])


In [12]:
attn_weights = torch.softmax( attn_scores , dim =-1)
print(attn_weights)

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])


In [13]:
all_context_vectors = attn_weights @ inputs
print(all_context_vectors)

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])
